## 1. 主题对齐：跨手型手内旋转泛化的核心矛盾

**用户反馈**：讨论主题确定为“跨手型手内旋转泛化”；当前处于框架/工程设计阶段，尚未进入 coding。

**分析**：

先把问题压缩成一句话：

> 目标不是“把旋转任务做出来”，而是“在不牺牲在线控制性的前提下，让策略跨 hand embodiment 迁移到 Leap family 之外”。

现有代表工作给出的边界大致如下：

| 工作 | 强项 | 关键缺口（对本题） |
|---|---|---|
| Patel & Song 2024 - GET-Zero | joint-token + 图偏置 attention，在线策略形态成熟 | 泛化主要在 Leap family 内；embodiment 编码偏弱（joint limits/几何/驱动能力表达不足）；训练范式依赖 per-expert distillation |
| Yang 等 2024 - AnyRotate | 单手型旋转任务做得深，含 sim-to-real 经验 | 不是跨手型泛化方法论文 |
| Liu 等 2025 - DexNDM | 现实差距建模（动力学）能力强 | 主要是 sim2real 方向，不直接解决跨手型表示问题 |
| Fei 等 2025 - T(R,O) Grasp | 关系建模强（edge-conditioned message/value） | 任务是抓取+扩散，不是在线手内旋转控制主干 |

因此，本题的方法不可替代性应写成：

1. **在线 joint policy 主干必须保留**（否则不适合高频控制）。
2. **embodiment 信息必须从“弱离散偏置”升级到“可学习连续结构条件”**（否则无法跨 family）。
3. **关系信息不能只决定看谁（routing），还要决定传什么（content）**，至少在指尖/接触相关边上要成立。

对应到你现有草图，最稳主线是：

- 前段：dynamic joint tokens × static embodiment tokens（sample-wise conditioning）
- 中段：relation-aware self-attention（先稳态版本，再加重版）
- 后段：local main path + relational residual head（保证控制保真）

从工程节奏看，MVP 不宜一开始把“跨 family hardest case + 全新训练范式 + 全重关系层”三件事同时上齐。更合理是先拍板“第一性验证目标”再定网络重度。

**小结**：当前最需要先收敛的，不是模块细节，而是“第一版要证明的泛化强度”。

**待确认**：第一版 MVP 你希望优先证明哪一级泛化强度（family 内、跨 family 轻量、还是跨 family 强泛化）？

## 2. 文献与实现精读：方法核提炼

**用户反馈**：先阅读 `papers/context.md` 中指定论文与实现代码，再继续讨论收敛。

**分析**：

我已完成以下必读文献与对应实现的梳理（选读项未展开）：

- **Qi 2022 - HORA**：核心是 rapid motor adaptation（从 proprio history 估计 object extrinsics），强调“在线适应”，但不解决跨手型结构建模。
- **Yang 2024 - AnyRotate**：核心是 dense tactile（contact pose + force）+ auxiliary goal（目标姿态推进），可覆盖多旋转轴与多手姿态，但默认是单手型路线。
- **Patel & Song 2024 - GET-Zero**：核心是 joint-token + graph-biased attention（SPD/parent/child bias）+ BC distillation + FK self-modeling；优势是在线控制结构成熟，短板是 embodiment 编码偏弱且泛化主要在 Leap family 内。
- **Fei 2025 - T(R,O) Grasp**：核心是 edge-conditioned value + grouped broadcast gate（代码层更明显），关系建模强、效率高，但任务形态是扩散抓取，不是在线旋转策略。
- **Liu 2025 - DexNDM**：核心是 joint-wise neural dynamics + residual action sim2real，强调用更可扩展的真实数据收集去拟合现实动力学偏差。

结合你指定代码实现，关键可迁移结论如下：

1. **GET 的主干可直接作为在线策略骨架**：
   - 实现中确实是“score bias 主导”的图编码（SPD/parent/child/edge embedding）
   - 这保证了在线推理形态与控制频率友好。

2. **TRO 的价值不在扩散本身，而在关系注入方式**：
   - 代码中体现了 grouped gate + edge-conditioned value 的组合，
   - 可迁移的是“边不仅决定看谁，也决定传什么”。

3. **DexNDM 更像第二阶段能力增强**：
   - 其 joint-wise dynamics residual 非常适合在你已有策略稳定后做 sim2real 闭环，
   - 不建议在第一版 MVP 就与跨手型方法创新同时推进。

基于“方法不可替代性”视角，可形成三档可执行路线：

| 路线 | 方法核 | 目标贡献 | 工程风险 |
|---|---|---|---|
| M0 | GET 主干 + 更强 embodiment token/edge（不引入重 gate） | 先证明跨手型泛化成立 | 低 |
| M1 | M0 + TRO 风格 grouped gate（做消融） | 证明关系建模增益机制 | 中 |
| M2 | M1 + DexNDM 风格 residual sim2real | 从泛化走向现实部署 | 高 |

**小结**：从论文叙事和工程可执行性看，第一篇最稳是 **M0 起步、M1 作为机制增强、M2 作为后续扩展**。

**待确认**：你希望第一版论文主线定在 M0、M1，还是直接追求 M2？

## 3. 任务重构：以工程可实现性为优先的设计分析

**用户反馈**：希望将问题从“网络方案争议”重构为“可分阶段推进的工程设计任务”，按 `资产生成与特征提取 / 数据集构建 / 网络架构` 三板块分析，重点解决 URDF 资产泛化表示与跨手型结构特征提取。

**分析**：

### 一、问题重述（为什么当前瓶颈不是网络细节）

当前最核心瓶颈不是 `plan` 中某个 attention 变体，而是：

1. **资产语义不统一**：不同 hand family 的 URDF 命名、关节语义、碰撞定义风格差异很大。
2. **结构特征定义未闭环**：尚未形成“可自动提取、跨手型一致、可喂给网络”的统一特征协议。
3. **工程链路缺少中间层**：直接在原始 URDF 上做网络输入，导致资产生成、数据构建、模型设计强耦合。

所以真正问题应重述为：

> 先建立一个 **比 URDF 更抽象、但可落地反向生成 URDF** 的中间表示，再围绕它定义稳定的结构特征协议与数据协议，最后再比较网络方案。

---

### 二、三大板块优先级判断

| 板块 | 是否前置依赖 | 当前关键卡点 | 若不先解决的后果 |
|---|---|---|---|
| 资产生成与特征提取 | **是（最高优先级）** | 缺少统一中间表示；collision 与骨架对齐规则不统一 | 后续数据/网络输入频繁返工，实验不可复现 |
| 数据集构建 | 中（依赖资产） | 无统一 manifest 与 split 规则；家族间公平性不足 | 结论不可信，难证明“跨手型”而非“过拟合某家族” |
| 网络架构 | 后置（依赖前两者） | 输入特征定义不稳定，架构对比会被数据偏差污染 | 无法得出有效 ablation，论文叙事失焦 |

结论：**先资产与特征协议，再数据协议，最后网络细化。**

---

### 三、Q1 设计建议：广义灵巧手中间表示（HIR, Hand Intermediate Representation）

建议定义 `HIR v0.1`，作为 `URDF <-> 学习系统` 的中间层。

#### 3.1 分层结构

| 层级 | 核心字段 | 作用 |
|---|---|---|
| Hand-level | hand_id, family, handedness, root_frame, palm_layout | 统一整手元信息 |
| Finger-level | finger_id, base_pose_on_palm, chain_type | 描述布指方式与手指拓扑 |
| Joint-level | joint_type, axis_local, origin(parent->joint), limits, drive params | 运动学主语义 |
| Link-level | inertial, visual list, **collision list(多体元)** | 几何与动力学语义 |
| Tip-level | tip_frame, tip_role, contact_patch descriptor, sensor flag | 接触相关关键语义 |
| Derived Graph-level | parent/child, chain index, SPD, symmetry group | 供网络直接使用的结构图特征 |

#### 3.2 映射到 URDF 的策略

- **HIR -> URDF**：确定性序列化（固定命名规则 + 固定字段顺序）。
- **URDF -> HIR**：容错解析（允许 mesh-only、primitive-only、mixed collision）。
- 对 visual/collision 采用显式策略位：
  - `strict_align`: visual 与 collision 都随参数变化同步更新。
  - `collision_first`: 训练阶段优先 collision 准确，visual 可延后修复。

#### 3.3 与现有资产兼容性

- **Get-Zero 路线**：其脚本已证明“拓扑枚举 + 长度改造 + YAML 元数据”可自动化，适合作为 HIR 生成器雏形。
- **Allegro/Shadow 资产**：可先做 `URDF -> HIR` 兼容导入，不要求一开始就同等参数化自由度。
- **TRO-Grasp 变体资产**：可作为跨 family 数据源，验证 HIR 的表达覆盖能力。

#### 3.4 参数化与人工修订边界

- **优先参数化（自动化）**：关节拓扑、joint origin/axis/limit、primitive collision 参数、link 尺度。
- **允许人工修订（阶段性）**：复杂 mesh 的高保真改造、特殊 tip 几何精修、视觉材质一致性。

```mermaid
flowchart LR
    A[现有 URDF 资产] --> B[URDF->HIR 解析器]
    B --> C[HIR Canonical Schema]
    C --> D[HIR 变体生成器]
    D --> E[HIR->URDF 生成器]
    E --> F[Isaac URDF->USD 导入]
    C --> G[结构特征提取器]
    G --> H[训练数据 Manifest 与 Token 数据]
```

---

### 四、Q2 特征设计建议：跨手型结构特征协议

现有 `joint / link / tip` 三分法是合理的，但建议补两层：`finger-level` 与 `hand-level`。

#### 4.1 最小可行特征集（MVP）

| 层级 | MVP 必须特征 | 建议编码 |
|---|---|---|
| Joint | type, axis, origin xyz/rpy, limits, parent/child index | 标量归一化 + 类别 embedding |
| Link | collision 几何类型(box/capsule/sphere/mesh), 尺度参数, collision 相对位姿 | 混合编码（类别+连续量） |
| Tip | is_tip, tip_frame offset, tip collision 主尺度 | 二值 + 连续量 |
| Finger | finger_id, chain length, chain depth | 类别 embedding + 标量 |
| Hand | palm 尺度统计, finger count, symmetry tags | 全局向量 |
| Edge | parent-child, SPD, same-finger/cross-finger | 图边离散特征 |

#### 4.2 增强版特征集（后续扩展）

- inertia/mass/compliance/friction/actuator limits。
- mesh 几何嵌入（BPS/点云统计/凸分解摘要）。
- tip 接触区域统计（曲率、法向主方向、patch 面积）。
- 手级拓扑签名（对称群、掌面分布统计）。

#### 4.3 对当前实现的关键修正建议

现有 Get-Zero `get_geometry_properties` 的抽取更偏 **joint+visual**，且有 `每个 link 仅一个 visual` 的假设。若目标是跨手型泛化，建议升级为：

1. **collision-first 抽取**（先 collision 再 visual），
2. 支持 `一个 link 多 collision 几何体`，
3. 将 tip 作为一等语义对象（而非仅靠 link 名称推断）。

---

### 五、工程落地建议（建议推进顺序）

1. **定义 HIR v0.1 schema + 命名与坐标约定文档**（先冻结字段，不急着全覆盖）。
2. **实现 URDF -> HIR 解析器**，先覆盖 Leap + Allegro + Shadow 三族样本。
3. **实现 HIR -> URDF 生成器（MVP）**，仅支持 `box/capsule/sphere/mesh` 四类 collision。
4. **实现 HIR -> 结构特征提取器**，产出标准化 token/graph 输入与 manifest。
5. **构建小规模验证闭环**：2-3 family、有限变体、固定 split，先验证“特征协议可训练”。

---

### 六、风险与取舍

| 风险 | 触发方式 | 建议折中 |
|---|---|---|
| 过度追求通用性导致实现爆炸 | 一开始支持所有几何与所有家族差异 | 先做 HIR v0.1（80/20），可扩展优先 |
| collision mesh 过于自由导致难规范 | mesh-only 资产无统一尺度/坐标习惯 | 先统一 collision-first 规则 + mesh 元数据校验 |
| 不同 family 关节语义不一致 | 同名关节语义不同、轴定义不同 | 引入显式 semantic tag，不依赖名称猜测 |
| 特征过细难学，过粗丢信息 | 一次性塞入大量物理细节 | 先 MVP 特征，增强特征走可控增量 ablation |
| 数据集评估失真 | split 不区分 family/构型难度 | 固定 family-level holdout + topology-level holdout 双协议 |

**小结**：当前最优策略是先把“资产表示与结构特征协议”独立成可执行工程任务，再让网络方案在稳定输入协议上做可解释比较。

**待确认**：下一轮你希望我先细化哪一项为“可直接开工的规格草案”？
1. HIR v0.1 字段规范（含示例）
2. URDF->HIR 解析规则（含容错与校验）
3. MVP 特征协议（含归一化与编码细则）
4. 数据集 split 与评测协议（family/topology 双 holdout）

## 4. 讨论方式校正：先调研结论，再谈执行

**用户反馈**：希望先做调研并给出明确看法，而不是直接推进“开工项”。

**分析**：

你这个要求是合理的。基于已完成调研（指定论文 + Get-Zero/TRO/Get-Zero distill 代码 + 现有 Allegro/Leap 资产），我先给出**明确立场**：

### 立场 1：先做“资产语义统一”，否则后续网络对比结论不可靠

- Get-Zero 证明了 URDF 可以程序化生成大规模变体，但它依赖一套特定命名/结构约束。
- 现有 `embodiment_util` 的几何抽取对 URDF 有若干强假设（例如 link visual 读取方式、语义字段固定），对跨 family 可扩展性有限。
- 因此在跨手型目标下，**先定义中间语义层**是必要条件，不是可选优化。

### 立场 2：必须以 collision 语义为主，不应以 visual 语义为主

- 你关心的接触建模与泛化本质上由 collision 几何决定。
- 现有资产中同一手型也存在 box/sphere/mesh 混用、origin 差异、tip 处理差异。
- 所以特征提取应优先保证 collision 可解释与可抽取；visual 可作为次级信息。

### 立场 3：当前阶段不宜继续细抠网络分支（A/B/C）

- 在输入协议未稳定前比较网络，会把资产偏差误判为模型差异。
- 更稳的节奏是：先把“可抽取且一致”的结构特征定义清楚，再做网络消融。

---

### 先调研、后推进的建议节奏（不是立刻开工）

本阶段先产出“调研结论包”，而不是实现代码：

1. **跨手型资产差异矩阵**（Leap / Allegro / Shadow）：命名、拓扑、collision 语义、tip 语义。
2. **现有提取链路缺口审计**：哪些字段现在可抽取、哪些抽取不稳定、哪些完全缺失。
3. **特征必要性分级**：必须有 / 推荐有 / 后续可选（并说明理由）。

这样你可以先判断方向是否对，再决定是否进入实现。

**小结**：你提的“先调研再输出观点”是正确顺序；我已给出当前三条核心判断，下一步先补齐证据化的差异矩阵与缺口审计。

**待确认**：下一轮你更希望我先展开哪份“调研结论包”？
1. 跨手型资产差异矩阵（结构/语义对比）
2. collision 可提取性与一致性审计
3. 现有代码链路缺口清单（按严重程度）

## 5. 互动方式纠偏（语气与节奏）

**用户反馈**：当前表达口气不符合讨论预期，需要改为更平和、学术化、非压迫式沟通。

**分析**：

这个反馈是有效的，后续讨论方式调整如下：

1. **先事实后判断**：先给证据，再给结论，不用推进式措辞。
2. **降低指令感**：减少“你要/你应当”表达，改为“可选路径 + 原因说明”。
3. **问题最小化**：每轮只问一个必要问题，避免连续施压式追问。
4. **讨论优先于执行**：在你明确同意前，不把讨论直接切换为“开工导向”。

在这个风格下，下一轮我会先给证据化结论，再请你做最小决策。

**小结**：后续采用“证据-分析-可选结论”的低压讨论模式。

**待确认**：是否继续按该风格推进“跨手型资产差异矩阵”的调研结论。

## 6. 跨手型资产差异矩阵（事实层）

**用户反馈**：同意先看差异矩阵（只给事实证据，不做推进）。

**分析**：

### 6.1 证据来源（本地文件）

- Get-Zero 资产生成脚本：`get_zero/get_zero/rl/scripts/gen_leap_assets.py`
- Get-Zero 结构提取：`get_zero/get_zero/distill/utils/embodiment_util.py`
- Leap（AnyMani）URDF：`AnyMani/source/anymani/assets/leap_hand_sim_urdf/leap_hand/robot.urdf`
- Allegro（hora）URDF：`hora/assets/allegro/allegro.urdf`
- Allegro（TRO）URDF：`TRO-Grasp/data/data_urdf/robot/allegro/allegro_hand_left.urdf`
- Shadow（TRO）URDF 族：`TRO-Grasp/data/data_urdf/robot/shadowhand/*.urdf`

### 6.2 差异矩阵

| 维度 | Leap（Get-Zero/AnyMani） | Allegro（hora/TRO） | Shadow（TRO） | 影响 |
|---|---|---|---|---|
| 命名风格 | 关节可为 `a_0...a_15` 或生成编号；link 多为功能名 | `joint_0.0`/`link_0.0` 风格，含 tip fixed link | 与 Allegro 类似但拓扑更复杂 | 不能依赖名字语义，需语义标签层 |
| 变体生成方式 | Get-Zero 有系统化脚本：拓扑枚举 + 长度变体 + YAML 元数据 | TRO 里有大量变体 URDF，但更像“结果集”而非统一生成脚本 | 同上 | 泛化实验可用，但工程复现链路不对称 |
| collision 表达 | Leap 常见 mesh collision（与 visual 同源） | hora 版本常见 box/sphere；TRO 版本为混合（box + mesh） | 多为 mesh/混合 | collision 类型分布跨 family 不一致 |
| visual/collision 对齐方式 | 依赖 origin 偏置与 mesh 文件约定 | 同一手型不同仓库也不一致 | 同上 | 对齐规则需要显式校验，不可隐式假设 |
| 拓扑元数据 | Get-Zero 会额外写 YAML（dofCount、joint map、sim-real 映射） | 普通 URDF 通常不带外部元数据 | 普通 URDF 通常不带外部元数据 | 若无中间层，数据构建会依赖“外部脚本习惯” |
| 解析器假设 | `embodiment_util` 可抽 joint/graph 特征，但几何抽取存在特定假设（例如 link visual 使用方式） | 面对多 collision/mixed 几何时信息颗粒度不足 | 同类问题 | 现有抽取链路可用但不够“跨 family 稳健” |

### 6.3 直接可见的共性与分歧

**共性**：
- 都能以 URDF 描述骨架与几何；
- 都可导出 parent-child 图结构并计算 SPD；
- 都能提取 joint limits/axis/origin 这类核心运动学字段。

**分歧**：
- collision 表示差异大（primitive-only / mesh-only / mixed）；
- 命名与 tip 表达差异大；
- 变体生成链路成熟度差异大（Get-Zero 强，其他家族弱）。

这说明“跨手型泛化”若要稳定推进，资产侧至少要先解决：

1. **命名无关的语义映射**（不依赖 joint/link 名字）；
2. **collision-first 的统一抽取规则**（支持多 collision 项）；
3. **元数据规范化**（把 DoF 映射、tip 语义、family 标签外显）。

**小结**：当前最大的跨 family 不一致，不在关节拓扑能否抽取，而在 collision 语义与元数据规范化。

**待确认**：下一轮你希望我先给哪一份“证据化补充”？
1. collision 可提取性审计（按几何类型逐项）
2. 现有解析链路缺口清单（按风险等级）

## 7. 关键困惑澄清：置换等变性与 collision-first 编码

**用户反馈**：
1) 命名无关语义映射是否可通过“置换不变/等变”来实现？会不会与图结构偏置冲突？
2) collision-first 抽取是否可直接复用 TRO 类编码器？

**分析**：

### 7.1 关于“命名无关”的正确目标：不是全不变，而是“重标号等变”

你直觉是对的，但目标应更精确：

- 不是要求模型对节点置换后输出完全不变（invariant）；
- 而是要求在节点重标号后，输出随同重排（equivariant）。

数学上可写为：若节点重排矩阵为 $P$，则希望

$$
\text{Policy}(P X,\; P A P^\top,\; P E P^\top) = P\,\text{Policy}(X,A,E)
$$

其中 $X$ 是节点特征，$A$ 是邻接/拓扑，$E$ 是边特征。

这意味着：
- **索引号本身不应承载语义**；
- 语义应来自图关系（父子/SPD/边几何）和局部物理特征（joint/link/tip）。

#### 会和图偏置冲突吗？

一般不会，前提是图偏置本身来自关系张量（如 SPD、parent/child 距离、相对变换），而不是固定位置索引。

真正会破坏等变性的做法是：

1. 使用固定 index 位置编码（按 token 序号给绝对编码）；
2. 在 head 中硬编码“第 k 个 token = 第 k 个电机语义”；
3. 用名称字符串直接映射语义而不做标准化标签层。

#### 工程上可采用的折中

- 内部网络尽量关系驱动（graph-aware equivariant）；
- 输出阶段再通过 `node_id -> actuator_id` 显式映射到控制接口；
- 允许加入语义标签（如 `is_thumb`, `is_tip`），但这些应来自结构推导而非名字。

---

### 7.2 关于 collision-first：TRO 思路可借，但不能直接“拿来即用”

结论先说：

- **没有现成可直接替换的 hand-collision 编码器**；
- 但 TRO 的建模思想可以直接迁移。

TRO 可复用的关键不是“扩散”，而是：

1. 用几何 token 表示实体（object patch / link）；
2. 用相对变换边表示关系；
3. 用关系注意力在图上做信息传递。

#### 对 hand 侧的对应迁移

- 将每个 link 的多个 collision 几何体视为一个小集合：
  - primitive（box/capsule/sphere）直接参数化；
  - mesh 用采样点云后做 BPS/统计嵌入。
- 先在 link 内做“集合聚合”得到 link-level collision embedding；
- 再进入 hand-level graph（joint/link/tip）做关系传播。

可分三阶段：

1. **MVP**：只用 primitive 参数 + mesh 的 AABB/体积/惯性近似统计；
2. **增强**：mesh 点云采样 + BPS 嵌入；
3. **高级**：collision item 级 token + 层级图（item->link->hand）。

这样可以先保证可自动抽取，再逐步提升几何表达力。

---

### 7.3 阶段性结论（避免继续混乱）

1. 你要的“命名无关语义映射”本质是 **重标号等变**，不是“完全置换不变”。
2. 图结构偏置与等变不冲突，冲突点在“绝对索引编码”和“名字硬绑定语义”。
3. collision-first 可以借 TRO 思路，但需做 hand-specific 的分层编码，不是直接复用 TRO 代码。

**小结**：这两个点都能落地，关键是先把目标从“抽象不变性”改写成“可验证的等变性约束 + collision 分层编码协议”。

**待确认**：下一轮你更希望我先给哪一个“最小验证方案”？
1. 等变性验证方案（如何实验验证“命名无关”是否成立）
2. collision-first 编码方案（MVP 字段 + 聚合流程草图）

## 8. 聚焦资产：多样化手资产的定义与生成框架（不展开网络）

**用户反馈**：当前更关心“如何定义和生成多样化手资产”，网络部分可以后置。

**分析**：

很好，这样聚焦更有效。下面只讨论资产定义与生成。

### 8.1 先定义“我们到底要生成什么”

建议把一只手资产定义为：

$$
\mathcal{H} = (\mathcal{T}_{kin},\ \mathcal{G}_{coll},\ \mathcal{P}_{dyn},\ \mathcal{M}_{meta})
$$

其中：
- $\mathcal{T}_{kin}$：运动学树（palm/finger/joint/link 拓扑与位姿参数）
- $\mathcal{G}_{coll}$：碰撞几何集合（primitive + mesh）
- $\mathcal{P}_{dyn}$：动力学与驱动参数（质量、惯量、limit、effort 等）
- $\mathcal{M}_{meta}$：语义元数据（tip 标签、finger 角色、映射关系）

目标不是“生成尽可能多 URDF 文件”，而是生成**满足约束且可训练可比较**的资产族。

---

### 8.2 生成参数空间（建议分 5 组）

| 参数组 | 典型变量 | 备注 |
|---|---|---|
| 拓扑参数 | 手指数、每指关节数、thumb 接入位、是否保留侧摆关节 | 决定结构多样性上限 |
| 运动学参数 | joint origin、axis、link 长度比例、关节限位 | 决定可达空间与可操纵性 |
| 几何参数 | collision 类型、尺寸、mesh 变体、tip 几何 | 决定接触行为与仿真稳定性 |
| 动力学参数 | mass/inertia、阻尼、摩擦、effort/velocity 上限 | 决定控制难度与 sim2real 敏感性 |
| 语义参数 | finger role、tip role、对称组、actuator 映射 | 决定跨 family 一致性 |

---

### 8.3 资产生成算子（Mutation Operators）

建议把“生成”拆成可组合算子，而不是一次性大脚本：

1. `TopoMutate`：增删关节/连杆、调整 finger 布局。
2. `KineMutate`：调整 axis/origin/length/limit。
3. `GeomMutate`：替换或参数化 collision（box/capsule/sphere/mesh）。
4. `DynMutate`：质量惯量与驱动参数扰动。
5. `SemAnnotate`：自动补充 tip/finger/对称组等语义标签。

每个算子都应附带约束检查：
- 树结构合法（无环、单根、父子一致）；
- 关节轴与限位合法；
- collision 有效（非负尺寸、文件存在、位姿可解析）；
- 可导入 Isaac（URDF parser smoke test）。

---

### 8.4 生成流程（建议）

```text
Template Hand(s)
  -> Canonical Intermediate Representation
  -> Apply Mutation Operators (with constraints)
  -> Emit URDF + Metadata
  -> Auto Validation (topology / parser / kinematic sanity)
  -> Registry + Split Assignment
```

其中 `Registry` 至少应记录：
- 资产 ID、父资产 ID（追踪变体来源）
- 变异算子与参数（可复现）
- family 标签与难度标签（用于后续 split）

---

### 8.5 第一阶段建议（只做资产，不碰网络）

**Phase A（最小可行）**：
- 只支持 4 类 collision：box/capsule/sphere/mesh；
- 先做 2 个 family（Leap + Allegro）；
- 每个 family 生成 50~100 个约束合法变体；
- 输出 URDF + 元数据清单 + 自动校验报告。

这一步完成后，你就会得到“可用资产生产线”，而不是一次性手工资产集合。

**小结**：多样化资产生成的核心不是“写一个大脚本”，而是先固定资产语义与算子体系，再做可约束、可追踪、可复现的批量生成。

**待确认**：下一轮你希望我先细化哪一块？
1. 生成参数空间的“可采样范围表”（每项给上下界）
2. 变异算子清单（输入/输出/约束）
3. 自动校验清单（生成后必须通过的检查）

## 9. 从讨论切到实施：MVP coding 任务包（文件级）

**用户反馈**：希望停止空谈，直接开始 `AnyMani/source/anymani/embodiment` 的最小实现，并先给可执行 plan。

**分析**：

已将最小实现计划写入 repo memory：`/memories/repo/embodiment_mvp_plan.md`。

为了直接进入 coding，建议先冻结下面这版**文件级最小任务包**（与 Get-Zero 风格对齐）：

### 9.1 建议目录（MVP）

- `source/anymani/embodiment/schema/hir_v01.py`
- `source/anymani/embodiment/io/urdf_to_hir.py`
- `source/anymani/embodiment/io/hir_to_urdf.py`
- `source/anymani/embodiment/mutate/topology.py`
- `source/anymani/embodiment/mutate/kinematics.py`
- `source/anymani/embodiment/mutate/geometry.py`
- `source/anymani/embodiment/validate/checks.py`
- `source/anymani/embodiment/cli/gen_assets.py`

### 9.2 每个模块最小职责

1. `hir_v01.py`：定义 Hand/Finger/Joint/Link/Collision/Tip 的 dataclass（必填字段）。
2. `urdf_to_hir.py`：解析 URDF 到 HIR（不依赖名字语义，按 parent-child 拓扑推断角色）。
3. `hir_to_urdf.py`：将 HIR 稳定序列化回 URDF。
4. `topology.py`：删关节/删链/手指数变异（带约束）。
5. `kinematics.py`：origin/axis/limit/length 变异（带上下界）。
6. `geometry.py`：collision 参数变异（先支持 box/capsule/sphere/mesh 引用）。
7. `checks.py`：结构合法性与字段完整性检查。
8. `gen_assets.py`：统一 CLI，输入模板 URDF，输出变体 URDF + metadata。

### 9.3 最小成功标准

- Leap + Allegro 各生成至少 10 个合法变体；
- 每个变体都有 metadata（dof、joint map、tip 列表）；
- 校验通过率 >= 95%。

**小结**：现在已经具备“可直接开始 coding”的最小任务边界，后续可以按该文件包落地。

**待确认**：是否以这版文件级任务包作为你要的最小实现起点。

## 10. 状态说明：从讨论到 coding 的切换准备

**用户反馈**：确认按最小任务包直接开干，可参考/复用 TRO 与 Get-Zero 资产脚本。

**分析**：

当前讨论已经收敛到可执行边界。由于本轮是讨论模式，先给出无缝切换方案，避免阻塞：

1. 输出首批文件的完整代码草案（按 9.1 的目录）。
2. 输出最小验证命令（生成 2 个 family、各 10 个变体）。
3. 输出失败排查清单（解析失败/拓扑非法/collision 缺失）。

这样你切到 coding 执行时，不需要再二次讨论需求。

**小结**：需求已收敛，下一步是“文件级代码草案 + 验证脚本”的交付。

## 11. 首批可落盘代码草案（Schema + 解析 + 校验）

**用户反馈**：先给代码草案（讨论模式内）。

**分析**：下面给第一批最小可行代码草案，目标是先打通 `URDF -> HIR -> 校验`。

---

### 文件 1：`source/anymani/embodiment/schema/hir_v01.py`

```python
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Dict, List, Literal, Optional, Tuple

GeomType = Literal["box", "sphere", "capsule", "mesh", "unknown"]
JointType = Literal["revolute", "prismatic", "fixed", "continuous", "unknown"]


@dataclass
class Pose:
    xyz: Tuple[float, float, float] = (0.0, 0.0, 0.0)
    rpy: Tuple[float, float, float] = (0.0, 0.0, 0.0)


@dataclass
class CollisionItem:
    geom_type: GeomType = "unknown"
    pose: Pose = field(default_factory=Pose)
    # primitive params
    size: Optional[Tuple[float, float, float]] = None      # box
    radius: Optional[float] = None                         # sphere/capsule
    length: Optional[float] = None                         # capsule
    # mesh params
    mesh_file: Optional[str] = None
    mesh_scale: Optional[Tuple[float, float, float]] = None


@dataclass
class VisualItem:
    geom_type: GeomType = "unknown"
    pose: Pose = field(default_factory=Pose)
    size: Optional[Tuple[float, float, float]] = None
    radius: Optional[float] = None
    length: Optional[float] = None
    mesh_file: Optional[str] = None
    mesh_scale: Optional[Tuple[float, float, float]] = None


@dataclass
class InertialItem:
    mass: Optional[float] = None
    pose: Pose = field(default_factory=Pose)
    # ixx, ixy, ixz, iyy, iyz, izz
    inertia: Optional[Tuple[float, float, float, float, float, float]] = None


@dataclass
class LinkSpec:
    link_id: str
    collisions: List[CollisionItem] = field(default_factory=list)
    visuals: List[VisualItem] = field(default_factory=list)
    inertial: Optional[InertialItem] = None


@dataclass
class JointSpec:
    joint_id: str
    joint_type: JointType = "unknown"
    parent_link: str = ""
    child_link: str = ""
    pose_parent_to_joint: Pose = field(default_factory=Pose)
    axis_local: Tuple[float, float, float] = (0.0, 0.0, 0.0)
    limit_lower: Optional[float] = None
    limit_upper: Optional[float] = None
    effort_limit: Optional[float] = None
    velocity_limit: Optional[float] = None


@dataclass
class FingerSpec:
    finger_id: str
    base_link: str
    chain_joint_ids: List[str] = field(default_factory=list)
    chain_link_ids: List[str] = field(default_factory=list)


@dataclass
class TipSpec:
    tip_link: str
    parent_link: Optional[str] = None
    tip_role: Optional[str] = None  # e.g. thumb_tip/index_tip/unknown_tip


@dataclass
class GraphDerived:
    root_link: Optional[str] = None
    dof_count: int = 0
    joint_name_to_joint_i: Dict[str, int] = field(default_factory=dict)
    parent_map: Dict[str, Optional[str]] = field(default_factory=dict)  # link -> parent link
    child_map: Dict[str, List[str]] = field(default_factory=dict)       # link -> child links


@dataclass
class HandHIR:
    hir_version: str = "0.1"
    hand_id: str = "unknown_hand"
    family: str = "unknown_family"
    handedness: Literal["left", "right", "unknown"] = "unknown"
    root_link: Optional[str] = None

    links: Dict[str, LinkSpec] = field(default_factory=dict)
    joints: Dict[str, JointSpec] = field(default_factory=dict)
    fingers: List[FingerSpec] = field(default_factory=list)
    tips: List[TipSpec] = field(default_factory=list)
    graph: GraphDerived = field(default_factory=GraphDerived)

    metadata: Dict[str, str] = field(default_factory=dict)

    def to_dict(self) -> dict:
        return asdict(self)
```

---

### 文件 2：`source/anymani/embodiment/io/urdf_to_hir.py`

```python
from __future__ import annotations

import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from anymani.embodiment.schema.hir_v01 import (
    CollisionItem,
    FingerSpec,
    GraphDerived,
    HandHIR,
    InertialItem,
    JointSpec,
    LinkSpec,
    Pose,
    TipSpec,
    VisualItem,
)


def _parse_vec3(text: Optional[str], default=(0.0, 0.0, 0.0)) -> Tuple[float, float, float]:
    if text is None:
        return default
    vals = [float(x) for x in text.strip().split()]
    if len(vals) != 3:
        return default
    return (vals[0], vals[1], vals[2])


def _parse_origin(elem: Optional[ET.Element]) -> Pose:
    if elem is None:
        return Pose()
    xyz = _parse_vec3(elem.attrib.get("xyz"), (0.0, 0.0, 0.0))
    rpy = _parse_vec3(elem.attrib.get("rpy"), (0.0, 0.0, 0.0))
    return Pose(xyz=xyz, rpy=rpy)


def _parse_mesh_scale(text: Optional[str]) -> Optional[Tuple[float, float, float]]:
    if text is None:
        return None
    vals = [float(x) for x in text.strip().split()]
    if len(vals) == 3:
        return (vals[0], vals[1], vals[2])
    return None


def _parse_collision_item(collision_elem: ET.Element) -> CollisionItem:
    pose = _parse_origin(collision_elem.find("origin"))
    geom = collision_elem.find("geometry")
    if geom is None:
        return CollisionItem(geom_type="unknown", pose=pose)

    box = geom.find("box")
    if box is not None:
        size = _parse_vec3(box.attrib.get("size"))
        return CollisionItem(geom_type="box", pose=pose, size=size)

    sphere = geom.find("sphere")
    if sphere is not None:
        radius = float(sphere.attrib.get("radius", "0"))
        return CollisionItem(geom_type="sphere", pose=pose, radius=radius)

    capsule = geom.find("capsule")
    if capsule is not None:
        radius = float(capsule.attrib.get("radius", "0"))
        length = float(capsule.attrib.get("length", "0"))
        return CollisionItem(geom_type="capsule", pose=pose, radius=radius, length=length)

    mesh = geom.find("mesh")
    if mesh is not None:
        return CollisionItem(
            geom_type="mesh",
            pose=pose,
            mesh_file=mesh.attrib.get("filename"),
            mesh_scale=_parse_mesh_scale(mesh.attrib.get("scale")),
        )

    return CollisionItem(geom_type="unknown", pose=pose)


def _parse_visual_item(visual_elem: ET.Element) -> VisualItem:
    pose = _parse_origin(visual_elem.find("origin"))
    geom = visual_elem.find("geometry")
    if geom is None:
        return VisualItem(geom_type="unknown", pose=pose)

    box = geom.find("box")
    if box is not None:
        size = _parse_vec3(box.attrib.get("size"))
        return VisualItem(geom_type="box", pose=pose, size=size)

    sphere = geom.find("sphere")
    if sphere is not None:
        radius = float(sphere.attrib.get("radius", "0"))
        return VisualItem(geom_type="sphere", pose=pose, radius=radius)

    capsule = geom.find("capsule")
    if capsule is not None:
        radius = float(capsule.attrib.get("radius", "0"))
        length = float(capsule.attrib.get("length", "0"))
        return VisualItem(geom_type="capsule", pose=pose, radius=radius, length=length)

    mesh = geom.find("mesh")
    if mesh is not None:
        return VisualItem(
            geom_type="mesh",
            pose=pose,
            mesh_file=mesh.attrib.get("filename"),
            mesh_scale=_parse_mesh_scale(mesh.attrib.get("scale")),
        )

    return VisualItem(geom_type="unknown", pose=pose)


def _parse_inertial(inertial_elem: Optional[ET.Element]) -> Optional[InertialItem]:
    if inertial_elem is None:
        return None
    pose = _parse_origin(inertial_elem.find("origin"))
    mass_elem = inertial_elem.find("mass")
    mass = None if mass_elem is None else float(mass_elem.attrib.get("value", "0"))

    inertia_elem = inertial_elem.find("inertia")
    inertia = None
    if inertia_elem is not None:
        inertia = (
            float(inertia_elem.attrib.get("ixx", "0")),
            float(inertia_elem.attrib.get("ixy", "0")),
            float(inertia_elem.attrib.get("ixz", "0")),
            float(inertia_elem.attrib.get("iyy", "0")),
            float(inertia_elem.attrib.get("iyz", "0")),
            float(inertia_elem.attrib.get("izz", "0")),
        )

    return InertialItem(mass=mass, pose=pose, inertia=inertia)


def parse_urdf_to_hir(
    urdf_path: str,
    hand_id: Optional[str] = None,
    family: str = "unknown_family",
    handedness: str = "unknown",
) -> HandHIR:
    tree = ET.parse(str(urdf_path))
    root = tree.getroot()

    hir = HandHIR(
        hand_id=hand_id or Path(urdf_path).stem,
        family=family,
        handedness=handedness if handedness in ("left", "right", "unknown") else "unknown",
    )

    # parse links
    for link_elem in root.findall("link"):
        link_name = link_elem.attrib["name"]
        link_spec = LinkSpec(link_id=link_name)

        for c in link_elem.findall("collision"):
            link_spec.collisions.append(_parse_collision_item(c))
        for v in link_elem.findall("visual"):
            link_spec.visuals.append(_parse_visual_item(v))

        link_spec.inertial = _parse_inertial(link_elem.find("inertial"))
        hir.links[link_name] = link_spec

    # parse joints and build graph
    parent_map: Dict[str, Optional[str]] = {k: None for k in hir.links.keys()}
    child_map: Dict[str, List[str]] = {k: [] for k in hir.links.keys()}

    dof_joint_types = {"revolute", "prismatic", "continuous"}

    for joint_elem in root.findall("joint"):
        joint_name = joint_elem.attrib.get("name", "unknown_joint")
        joint_type = joint_elem.attrib.get("type", "unknown")

        parent_link_elem = joint_elem.find("parent")
        child_link_elem = joint_elem.find("child")
        if parent_link_elem is None or child_link_elem is None:
            continue

        parent_link = parent_link_elem.attrib.get("link", "")
        child_link = child_link_elem.attrib.get("link", "")

        axis_elem = joint_elem.find("axis")
        axis_local = _parse_vec3(axis_elem.attrib.get("xyz") if axis_elem is not None else None)

        limit_elem = joint_elem.find("limit")
        lower = float(limit_elem.attrib["lower"]) if (limit_elem is not None and "lower" in limit_elem.attrib) else None
        upper = float(limit_elem.attrib["upper"]) if (limit_elem is not None and "upper" in limit_elem.attrib) else None
        effort = float(limit_elem.attrib["effort"]) if (limit_elem is not None and "effort" in limit_elem.attrib) else None
        velocity = float(limit_elem.attrib["velocity"]) if (limit_elem is not None and "velocity" in limit_elem.attrib) else None

        joint_spec = JointSpec(
            joint_id=joint_name,
            joint_type=joint_type if joint_type in ("revolute", "prismatic", "fixed", "continuous") else "unknown",
            parent_link=parent_link,
            child_link=child_link,
            pose_parent_to_joint=_parse_origin(joint_elem.find("origin")),
            axis_local=axis_local,
            limit_lower=lower,
            limit_upper=upper,
            effort_limit=effort,
            velocity_limit=velocity,
        )
        hir.joints[joint_name] = joint_spec

        if child_link in parent_map:
            parent_map[child_link] = parent_link
        if parent_link in child_map:
            child_map[parent_link].append(child_link)

    # root link inference
    roots = [link for link, p in parent_map.items() if p is None]
    hir.root_link = roots[0] if roots else None

    # dof mapping (name-agnostic control时可后续覆盖，此处先稳定排序)
    dof_joint_names = [j.joint_id for j in hir.joints.values() if j.joint_type in dof_joint_types]
    dof_joint_names.sort()
    joint_name_to_joint_i = {name: i for i, name in enumerate(dof_joint_names)}

    # leaf links -> tip
    leaf_links = [link for link, children in child_map.items() if len(children) == 0]
    hir.tips = [TipSpec(tip_link=lk, parent_link=parent_map.get(lk), tip_role="unknown_tip") for lk in leaf_links]

    # finger inference（最简版本）：从 root 的直接子链出发分解串链
    fingers: List[FingerSpec] = []
    if hir.root_link is not None and hir.root_link in child_map:
        for i, base in enumerate(child_map[hir.root_link]):
            chain_links = [base]
            chain_joints: List[str] = []
            cur = base
            while True:
                next_children = child_map.get(cur, [])
                if len(next_children) != 1:
                    break
                nxt = next_children[0]
                # find connecting joint
                jname = None
                for j in hir.joints.values():
                    if j.parent_link == cur and j.child_link == nxt:
                        jname = j.joint_id
                        break
                if jname is not None:
                    chain_joints.append(jname)
                chain_links.append(nxt)
                cur = nxt
            fingers.append(FingerSpec(finger_id=f"finger_{i}", base_link=base, chain_joint_ids=chain_joints, chain_link_ids=chain_links))

    hir.fingers = fingers

    hir.graph = GraphDerived(
        root_link=hir.root_link,
        dof_count=len(joint_name_to_joint_i),
        joint_name_to_joint_i=joint_name_to_joint_i,
        parent_map=parent_map,
        child_map=child_map,
    )

    return hir
```

---

### 文件 3：`source/anymani/embodiment/validate/checks.py`

```python
from __future__ import annotations

from typing import List, Set

from anymani.embodiment.schema.hir_v01 import HandHIR


def validate_hir_basic(hir: HandHIR) -> List[str]:
    errs: List[str] = []

    if hir.root_link is None:
        errs.append("root_link is None")

    # link existence
    for j in hir.joints.values():
        if j.parent_link not in hir.links:
            errs.append(f"joint {j.joint_id}: parent_link '{j.parent_link}' not found")
        if j.child_link not in hir.links:
            errs.append(f"joint {j.joint_id}: child_link '{j.child_link}' not found")

    # axis validity for movable joints
    for j in hir.joints.values():
        if j.joint_type in ("revolute", "prismatic", "continuous"):
            ax = j.axis_local
            norm = (ax[0] ** 2 + ax[1] ** 2 + ax[2] ** 2) ** 0.5
            if norm < 1e-8:
                errs.append(f"joint {j.joint_id}: movable but axis norm is zero")

    # limit sanity
    for j in hir.joints.values():
        if j.limit_lower is not None and j.limit_upper is not None:
            if j.limit_lower > j.limit_upper:
                errs.append(f"joint {j.joint_id}: lower > upper")

    # collision sanity
    for lk, lspec in hir.links.items():
        for idx, c in enumerate(lspec.collisions):
            if c.geom_type == "box":
                if c.size is None or any(v <= 0 for v in c.size):
                    errs.append(f"link {lk} collision[{idx}] invalid box size")
            elif c.geom_type == "sphere":
                if c.radius is None or c.radius <= 0:
                    errs.append(f"link {lk} collision[{idx}] invalid sphere radius")
            elif c.geom_type == "capsule":
                if c.radius is None or c.radius <= 0 or c.length is None or c.length <= 0:
                    errs.append(f"link {lk} collision[{idx}] invalid capsule params")
            elif c.geom_type == "mesh":
                if not c.mesh_file:
                    errs.append(f"link {lk} collision[{idx}] mesh missing filename")

    # graph loop check（link graph）
    if hir.root_link is not None and hir.graph.child_map:
        visited: Set[str] = set()
        stack: Set[str] = set()

        def dfs(u: str):
            if u in stack:
                errs.append(f"cycle detected at link {u}")
                return
            if u in visited:
                return
            visited.add(u)
            stack.add(u)
            for v in hir.graph.child_map.get(u, []):
                dfs(v)
            stack.remove(u)

        dfs(hir.root_link)

    # dof mapping consistency
    dof_joint_set = {j for j in hir.graph.joint_name_to_joint_i.keys()}
    movable_set = {j.joint_id for j in hir.joints.values() if j.joint_type in ("revolute", "prismatic", "continuous")}
    if dof_joint_set != movable_set:
        errs.append("joint_name_to_joint_i mismatch with movable joint set")

    return errs
```

---

**小结**：这三份草案能先把“资产语义层”跑起来，后续再补 `HIR->URDF` 与 `mutation` 即可形成最小闭环。

**待确认**：下一批我继续给你哪两个文件草案？
1. `hir_to_urdf.py` + `mutate/topology.py`
2. `mutate/kinematics.py` + `mutate/geometry.py`
3. `cli/gen_assets.py` + `validate/smoke_test.py`

## 12. 连续补齐：最小实现闭环草案（不再分步打断）

**用户反馈**：希望减少打断，直接推进。

**分析**：下面一次性补齐 `HIR->URDF`、`mutation`、`CLI` 草案。

---

### 文件 4：`source/anymani/embodiment/io/hir_to_urdf.py`

```python
from __future__ import annotations

import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Optional

from anymani.embodiment.schema.hir_v01 import HandHIR, Pose


def _fmt_vec3(v):
    return f"{v[0]} {v[1]} {v[2]}"


def _set_origin(parent: ET.Element, pose: Pose):
    ET.SubElement(parent, "origin", attrib={"xyz": _fmt_vec3(pose.xyz), "rpy": _fmt_vec3(pose.rpy)})


def _indent(elem: ET.Element, level: int = 0):
    i = "\n" + level * "  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for e in elem:
            _indent(e, level + 1)
        if not e.tail or not e.tail.strip():
            e.tail = i
    if level and (not elem.tail or not elem.tail.strip()):
        elem.tail = i


def emit_hir_to_urdf(hir: HandHIR, out_path: str):
    robot = ET.Element("robot", attrib={"name": hir.hand_id})

    # links
    for link_name, lspec in hir.links.items():
        link_elem = ET.SubElement(robot, "link", attrib={"name": link_name})

        if lspec.inertial is not None:
            inertial = ET.SubElement(link_elem, "inertial")
            _set_origin(inertial, lspec.inertial.pose)
            if lspec.inertial.mass is not None:
                ET.SubElement(inertial, "mass", attrib={"value": str(lspec.inertial.mass)})
            if lspec.inertial.inertia is not None:
                ixx, ixy, ixz, iyy, iyz, izz = lspec.inertial.inertia
                ET.SubElement(inertial, "inertia", attrib={
                    "ixx": str(ixx), "ixy": str(ixy), "ixz": str(ixz),
                    "iyy": str(iyy), "iyz": str(iyz), "izz": str(izz),
                })

        for v in lspec.visuals:
            vis = ET.SubElement(link_elem, "visual")
            _set_origin(vis, v.pose)
            g = ET.SubElement(vis, "geometry")
            if v.geom_type == "box" and v.size is not None:
                ET.SubElement(g, "box", attrib={"size": _fmt_vec3(v.size)})
            elif v.geom_type == "sphere" and v.radius is not None:
                ET.SubElement(g, "sphere", attrib={"radius": str(v.radius)})
            elif v.geom_type == "capsule" and v.radius is not None and v.length is not None:
                ET.SubElement(g, "capsule", attrib={"radius": str(v.radius), "length": str(v.length)})
            elif v.geom_type == "mesh" and v.mesh_file is not None:
                mesh_attr = {"filename": v.mesh_file}
                if v.mesh_scale is not None:
                    mesh_attr["scale"] = _fmt_vec3(v.mesh_scale)
                ET.SubElement(g, "mesh", attrib=mesh_attr)

        for c in lspec.collisions:
            col = ET.SubElement(link_elem, "collision")
            _set_origin(col, c.pose)
            g = ET.SubElement(col, "geometry")
            if c.geom_type == "box" and c.size is not None:
                ET.SubElement(g, "box", attrib={"size": _fmt_vec3(c.size)})
            elif c.geom_type == "sphere" and c.radius is not None:
                ET.SubElement(g, "sphere", attrib={"radius": str(c.radius)})
            elif c.geom_type == "capsule" and c.radius is not None and c.length is not None:
                ET.SubElement(g, "capsule", attrib={"radius": str(c.radius), "length": str(c.length)})
            elif c.geom_type == "mesh" and c.mesh_file is not None:
                mesh_attr = {"filename": c.mesh_file}
                if c.mesh_scale is not None:
                    mesh_attr["scale"] = _fmt_vec3(c.mesh_scale)
                ET.SubElement(g, "mesh", attrib=mesh_attr)

    # joints
    for jname, jspec in hir.joints.items():
        j = ET.SubElement(robot, "joint", attrib={"name": jname, "type": jspec.joint_type})
        ET.SubElement(j, "parent", attrib={"link": jspec.parent_link})
        ET.SubElement(j, "child", attrib={"link": jspec.child_link})
        _set_origin(j, jspec.pose_parent_to_joint)
        ET.SubElement(j, "axis", attrib={"xyz": _fmt_vec3(jspec.axis_local)})

        limit_attr = {}
        if jspec.limit_lower is not None:
            limit_attr["lower"] = str(jspec.limit_lower)
        if jspec.limit_upper is not None:
            limit_attr["upper"] = str(jspec.limit_upper)
        if jspec.effort_limit is not None:
            limit_attr["effort"] = str(jspec.effort_limit)
        if jspec.velocity_limit is not None:
            limit_attr["velocity"] = str(jspec.velocity_limit)
        if limit_attr:
            ET.SubElement(j, "limit", attrib=limit_attr)

    _indent(robot)
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    ET.ElementTree(robot).write(out, encoding="utf-8", xml_declaration=True)
```

---

### 文件 5：`source/anymani/embodiment/mutate/topology.py`

```python
from __future__ import annotations

from copy import deepcopy
from typing import Optional

from anymani.embodiment.schema.hir_v01 import HandHIR


def _reindex_dof(hir: HandHIR):
    movable = [j for j in hir.joints.values() if j.joint_type in ("revolute", "prismatic", "continuous")]
    names = sorted([j.joint_id for j in movable])
    hir.graph.joint_name_to_joint_i = {n: i for i, n in enumerate(names)}
    hir.graph.dof_count = len(names)


def drop_last_joint_of_finger(hir: HandHIR, finger_id: str) -> HandHIR:
    out = deepcopy(hir)
    target = None
    for f in out.fingers:
        if f.finger_id == finger_id:
            target = f
            break
    if target is None or len(target.chain_joint_ids) == 0:
        return out

    # remove last joint and its child link (leaf)
    jname = target.chain_joint_ids[-1]
    js = out.joints.get(jname)
    if js is None:
        return out

    leaf_link = js.child_link
    parent_link = js.parent_link

    out.joints.pop(jname, None)
    out.links.pop(leaf_link, None)

    # graph maps
    if parent_link in out.graph.child_map:
        out.graph.child_map[parent_link] = [x for x in out.graph.child_map[parent_link] if x != leaf_link]
    out.graph.parent_map.pop(leaf_link, None)
    out.graph.child_map.pop(leaf_link, None)

    # finger chain shrink
    target.chain_joint_ids = target.chain_joint_ids[:-1]
    if target.chain_link_ids and target.chain_link_ids[-1] == leaf_link:
        target.chain_link_ids = target.chain_link_ids[:-1]

    # tips refresh
    out.tips = [t for t in out.tips if t.tip_link != leaf_link]

    _reindex_dof(out)
    return out
```

---

### 文件 6：`source/anymani/embodiment/mutate/kinematics.py`

```python
from __future__ import annotations

from copy import deepcopy

from anymani.embodiment.schema.hir_v01 import HandHIR


def scale_finger_origins(hir: HandHIR, finger_id: str, z_scale: float = 1.1) -> HandHIR:
    out = deepcopy(hir)
    f = next((x for x in out.fingers if x.finger_id == finger_id), None)
    if f is None:
        return out

    for jn in f.chain_joint_ids:
        j = out.joints.get(jn)
        if j is None:
            continue
        x, y, z = j.pose_parent_to_joint.xyz
        j.pose_parent_to_joint.xyz = (x, y, z * z_scale)
    return out


def widen_joint_limits(hir: HandHIR, ratio: float = 1.05, max_abs: float = 3.14) -> HandHIR:
    out = deepcopy(hir)
    for j in out.joints.values():
        if j.limit_lower is None or j.limit_upper is None:
            continue
        c = 0.5 * (j.limit_lower + j.limit_upper)
        h = 0.5 * (j.limit_upper - j.limit_lower) * ratio
        lo = max(-max_abs, c - h)
        hi = min(max_abs, c + h)
        if lo < hi:
            j.limit_lower, j.limit_upper = lo, hi
    return out
```

---

### 文件 7：`source/anymani/embodiment/mutate/geometry.py`

```python
from __future__ import annotations

from copy import deepcopy
from typing import Tuple

from anymani.embodiment.schema.hir_v01 import HandHIR


def scale_box_collisions_of_link(hir: HandHIR, link_id: str, scale: Tuple[float, float, float]) -> HandHIR:
    out = deepcopy(hir)
    l = out.links.get(link_id)
    if l is None:
        return out

    sx, sy, sz = scale
    for c in l.collisions:
        if c.geom_type == "box" and c.size is not None:
            x, y, z = c.size
            nx, ny, nz = x * sx, y * sy, z * sz
            if nx > 0 and ny > 0 and nz > 0:
                c.size = (nx, ny, nz)
    return out


def mesh_to_box_proxy(hir: HandHIR, link_id: str, proxy_size=(0.01, 0.01, 0.01)) -> HandHIR:
    out = deepcopy(hir)
    l = out.links.get(link_id)
    if l is None:
        return out

    for c in l.collisions:
        if c.geom_type == "mesh":
            c.geom_type = "box"
            c.mesh_file = None
            c.mesh_scale = None
            c.size = proxy_size
            c.radius = None
            c.length = None
    return out
```

---

### 文件 8：`source/anymani/embodiment/cli/gen_assets.py`

```python
from __future__ import annotations

import argparse
import json
from pathlib import Path

from anymani.embodiment.io.urdf_to_hir import parse_urdf_to_hir
from anymani.embodiment.io.hir_to_urdf import emit_hir_to_urdf
from anymani.embodiment.mutate.topology import drop_last_joint_of_finger
from anymani.embodiment.mutate.kinematics import scale_finger_origins, widen_joint_limits
from anymani.embodiment.mutate.geometry import scale_box_collisions_of_link
from anymani.embodiment.validate.checks import validate_hir_basic


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_urdf", required=True)
    parser.add_argument("--out_dir", required=True)
    parser.add_argument("--family", default="unknown_family")
    parser.add_argument("--count", type=int, default=10)
    args = parser.parse_args()

    out_dir = Path(args.out_dir)
    urdf_dir = out_dir / "urdf"
    meta_dir = out_dir / "meta"
    urdf_dir.mkdir(parents=True, exist_ok=True)
    meta_dir.mkdir(parents=True, exist_ok=True)

    base = parse_urdf_to_hir(args.input_urdf, family=args.family)

    for i in range(args.count):
        cur = base

        # minimal deterministic mutation mix
        if cur.fingers:
            cur = drop_last_joint_of_finger(cur, cur.fingers[0].finger_id)
            cur = scale_finger_origins(cur, cur.fingers[0].finger_id, z_scale=1.0 + 0.01 * (i % 3))
        cur = widen_joint_limits(cur, ratio=1.0 + 0.01 * (i % 2))

        # optional geometry mutation on first link
        if cur.links:
            first_link = list(cur.links.keys())[0]
            cur = scale_box_collisions_of_link(cur, first_link, (1.0, 1.0, 1.0))

        errs = validate_hir_basic(cur)
        if errs:
            print(f"[skip] sample {i:03d}: {errs[:2]}")
            continue

        sid = f"{i:03d}"
        emit_hir_to_urdf(cur, str(urdf_dir / f"{sid}.urdf"))

        with open(meta_dir / f"{sid}.json", "w", encoding="utf-8") as f:
            json.dump({
                "sample_id": sid,
                "family": cur.family,
                "hand_id": cur.hand_id,
                "dof_count": cur.graph.dof_count,
                "joint_name_to_joint_i": cur.graph.joint_name_to_joint_i,
                "tips": [t.tip_link for t in cur.tips],
            }, f, ensure_ascii=False, indent=2)

        print(f"[ok] {sid}")


if __name__ == "__main__":
    main()
```

---

### 建议的最小验证命令（草案）

```bash
python -m anymani.embodiment.cli.gen_assets \
  --input_urdf AnyMani/source/anymani/assets/leap_hand_sim_urdf/leap_hand/robot.urdf \
  --out_dir AnyMani/source/anymani/embodiment/_out/leap_mvp \
  --family leap \
  --count 10
```

```bash
python -m anymani.embodiment.cli.gen_assets \
  --input_urdf hora/assets/allegro/allegro.urdf \
  --out_dir AnyMani/source/anymani/embodiment/_out/allegro_mvp \
  --family allegro \
  --count 10
```

**小结**：现在最小闭环草案已经完整：`URDF->HIR`、`HIR->URDF`、`mutation`、`validate`、`CLI`。

**待确认**：下一轮我可以继续把这批草案整理成“按文件可直接复制”的最终版（补齐 import 路径和包初始化建议）。

## 13. 可复制落地版：包结构、导入与运行顺序

**用户反馈**：继续自动整理，不频繁提问。

**分析**：以下是把前述草案真正落盘时的最小注意点。

### 13.1 建议目录树（MVP）

```text
source/anymani/embodiment/
  __init__.py
  schema/
    __init__.py
    hir_v01.py
  io/
    __init__.py
    urdf_to_hir.py
    hir_to_urdf.py
  mutate/
    __init__.py
    topology.py
    kinematics.py
    geometry.py
  validate/
    __init__.py
    checks.py
  cli/
    __init__.py
    gen_assets.py
```

### 13.2 导入路径约定

- 如果项目包根是 `anymani`，则统一使用：
  - `from anymani.embodiment.schema.hir_v01 import ...`
- 如果你临时用脚本直接运行，建议用 `python -m ...` 方式，避免相对导入报错。

### 13.3 第一轮落盘顺序（建议）

1. 先落 `schema/hir_v01.py`。
2. 再落 `io/urdf_to_hir.py` 与 `validate/checks.py`。
3. 用一个临时脚本先跑 `parse_urdf_to_hir + validate`。
4. 再落 `io/hir_to_urdf.py`，做 round-trip（读 URDF -> HIR -> 写 URDF）。
5. 最后落 `mutate/*` 与 `cli/gen_assets.py`。

### 13.4 第一轮必须看到的产物

- `AnyMani/source/anymani/embodiment/_out/leap_mvp/urdf/*.urdf`
- `AnyMani/source/anymani/embodiment/_out/leap_mvp/meta/*.json`
- `AnyMani/source/anymani/embodiment/_out/allegro_mvp/urdf/*.urdf`
- `AnyMani/source/anymani/embodiment/_out/allegro_mvp/meta/*.json`

### 13.5 第一轮高概率报错点（提前规避）

1. **URDF 含未实现几何类型**（如 cylinder）：
   - 先在 parser 中映射到 `unknown` 并给 warning，不要直接抛异常。
2. **joint limit 缺字段**：
   - 对 `fixed` 或部分 URDF 的缺失字段允许 `None`。
3. **link 无 collision**：
   - 允许存在，但在校验中记为 warning 级别。
4. **根链接推断不唯一**：
   - 若多个 root，先按“被 joint.child 引用次数最少”选主 root，并记录 warning。

### 13.6 建议追加的最小 smoke test（下一步）

- Test A：Leap 模板读写 round-trip 不崩溃。
- Test B：Allegro 模板读写 round-trip 不崩溃。
- Test C：每个样本 `dof_count == len(joint_name_to_joint_i)`。
- Test D：每个 tip link 都是 `child_map` 叶子节点。

**小结**：到这一步，已经具备“可直接落盘并开始看到资产生成结果”的完整最小包。

## 14. 测试文件草案（最小回归）

**用户反馈**：继续补测试文件草案。

**分析**：先给两份最小测试：`round-trip` 与 `mutation`。

---

### 文件：`source/anymani/embodiment/tests/test_roundtrip.py`

```python
from __future__ import annotations

from pathlib import Path

from anymani.embodiment.io.urdf_to_hir import parse_urdf_to_hir
from anymani.embodiment.io.hir_to_urdf import emit_hir_to_urdf
from anymani.embodiment.validate.checks import validate_hir_basic


def test_roundtrip_leap(tmp_path: Path):
    src = Path("AnyMani/source/anymani/assets/leap_hand_sim_urdf/leap_hand/robot.urdf")
    assert src.exists(), f"missing test asset: {src}"

    hir = parse_urdf_to_hir(str(src), family="leap")
    errs = validate_hir_basic(hir)
    assert len(errs) == 0, f"validation failed: {errs[:3]}"

    out = tmp_path / "leap_roundtrip.urdf"
    emit_hir_to_urdf(hir, str(out))
    assert out.exists()

    # parse again to ensure generated file is readable
    hir2 = parse_urdf_to_hir(str(out), family="leap")
    errs2 = validate_hir_basic(hir2)
    assert len(errs2) == 0, f"roundtrip validation failed: {errs2[:3]}"


def test_roundtrip_allegro(tmp_path: Path):
    src = Path("hora/assets/allegro/allegro.urdf")
    assert src.exists(), f"missing test asset: {src}"

    hir = parse_urdf_to_hir(str(src), family="allegro")
    errs = validate_hir_basic(hir)
    assert len(errs) == 0, f"validation failed: {errs[:3]}"

    out = tmp_path / "allegro_roundtrip.urdf"
    emit_hir_to_urdf(hir, str(out))
    assert out.exists()
```

---

### 文件：`source/anymani/embodiment/tests/test_mutation.py`

```python
from __future__ import annotations

from pathlib import Path

from anymani.embodiment.io.urdf_to_hir import parse_urdf_to_hir
from anymani.embodiment.mutate.topology import drop_last_joint_of_finger
from anymani.embodiment.mutate.kinematics import scale_finger_origins, widen_joint_limits
from anymani.embodiment.mutate.geometry import scale_box_collisions_of_link
from anymani.embodiment.validate.checks import validate_hir_basic


def test_topology_mutation_keeps_validity():
    src = Path("AnyMani/source/anymani/assets/leap_hand_sim_urdf/leap_hand/robot.urdf")
    hir = parse_urdf_to_hir(str(src), family="leap")
    if not hir.fingers:
        return

    out = drop_last_joint_of_finger(hir, hir.fingers[0].finger_id)
    errs = validate_hir_basic(out)
    assert len(errs) == 0, f"topology mutation invalid: {errs[:3]}"


def test_kinematics_mutation_keeps_validity():
    src = Path("AnyMani/source/anymani/assets/leap_hand_sim_urdf/leap_hand/robot.urdf")
    hir = parse_urdf_to_hir(str(src), family="leap")
    if not hir.fingers:
        return

    out = scale_finger_origins(hir, hir.fingers[0].finger_id, z_scale=1.05)
    out = widen_joint_limits(out, ratio=1.02)
    errs = validate_hir_basic(out)
    assert len(errs) == 0, f"kinematics mutation invalid: {errs[:3]}"


def test_geometry_mutation_keeps_validity():
    src = Path("hora/assets/allegro/allegro.urdf")
    hir = parse_urdf_to_hir(str(src), family="allegro")
    if not hir.links:
        return

    first_link = list(hir.links.keys())[0]
    out = scale_box_collisions_of_link(hir, first_link, (1.01, 1.0, 0.99))
    errs = validate_hir_basic(out)
    assert len(errs) == 0, f"geometry mutation invalid: {errs[:3]}"
```

---

### 建议运行命令（草案）

```bash
pytest -q source/anymani/embodiment/tests/test_roundtrip.py
pytest -q source/anymani/embodiment/tests/test_mutation.py
```

**小结**：这两份测试能先保证“可解析、可变异、可回写”的最小稳定性。

## 15. README 使用说明草案（最小实现）

**用户反馈**：补 README 后结束讨论。

**分析**：以下为可直接放入 `source/anymani/embodiment/README.md` 的草案。

```markdown
# Embodiment MVP (URDF -> HIR -> URDF)

本目录提供跨手型资产最小实现链路：

1. 解析 URDF 到统一中间表示 HIR。
2. 在 HIR 上执行基础变异（拓扑/运动学/几何）。
3. 回写 URDF 与 metadata。
4. 通过最小校验与测试保证稳定性。

## 目录

- `schema/hir_v01.py`: HIR 数据结构定义
- `io/urdf_to_hir.py`: URDF 解析器
- `io/hir_to_urdf.py`: URDF 生成器
- `mutate/topology.py`: 拓扑变异
- `mutate/kinematics.py`: 运动学变异
- `mutate/geometry.py`: 几何变异
- `validate/checks.py`: 基础校验
- `cli/gen_assets.py`: 批量生成入口
- `tests/`: 最小回归测试

## 快速开始

### 1) 生成 Leap 变体

```bash
python -m anymani.embodiment.cli.gen_assets \
  --input_urdf AnyMani/source/anymani/assets/leap_hand_sim_urdf/leap_hand/robot.urdf \
  --out_dir AnyMani/source/anymani/embodiment/_out/leap_mvp \
  --family leap \
  --count 10
```

### 2) 生成 Allegro 变体

```bash
python -m anymani.embodiment.cli.gen_assets \
  --input_urdf hora/assets/allegro/allegro.urdf \
  --out_dir AnyMani/source/anymani/embodiment/_out/allegro_mvp \
  --family allegro \
  --count 10
```

### 3) 运行测试

```bash
pytest -q source/anymani/embodiment/tests/test_roundtrip.py
pytest -q source/anymani/embodiment/tests/test_mutation.py
```

## 设计约束（MVP）

- 优先 `collision-first` 语义，不依赖 visual 命名语义。
- 几何先支持：`box/capsule/sphere/mesh`。
- 允许部分字段缺省，但必须通过 `validate_hir_basic`。

## 已知限制

- 当前 finger/tip 语义推断是最简版本，复杂拓扑需增强。
- mesh 仅保留路径与缩放，未做深度几何编码。
- 变异算子目前以“合法性优先”，尚未引入任务难度控制。
```

**小结**：到这里，讨论侧的最小实现交付包（代码草案 + 测试 + README）已完整。

## 16. GET-Zero × T(R,O) 跨手族表示迁移（概念压缩）

**用户反馈**：希望用 1–2 段话解释 GET-Zero 跨 family 失效的核心瓶颈，以及如何把 T(R,O) 的表示优势迁移到连续 RL；并产出 2 个下一轮关键问题。

**分析**：Patel & Song 2024 - GET-Zero 的强项是关节 token + 拓扑偏置（SPD/parent-child），但其具身编码主要停留在“关节拓扑关系”，对跨手族最敏感的连续几何与执行器语义（link 几何比例、关节限位/驱动能力、接触坐标系差异）表达不足。于是策略在 Leap 家族内可迁移，但到了 Allegro/Shadow 这类 out-of-family 时，会把“形态差异”误当成“任务状态变化”，在连续接触切换与稳定操控上明显失稳。

Fei 等 2025 - T(R,O) Grasp 的关键可迁移点是：用相对 SE(3) 变换边（而非绝对坐标）构建关系表示，使跨 embodiment 的几何关系更具坐标不变性。迁移到 GET 范式时，不必搬扩散框架本体，而应把其边表示迁入在线策略网络：在 GET 的图注意力里引入 $e_{ij}^{RR}=\log((T_i^R)^{-1}T_j^R)$ 与 $e_{ik}^{OR}=\log((T_i^O)^{-1}T_k^R)$ 作为关系边特征（至少用于 attention bias，进阶可进入 value modulation），从而让策略在连续 RL 中学习“跨手族共享的接触几何规律”。

**小结**：瓶颈是 GET-Zero 的“拓扑强、几何弱”；修复方向是把 T(R,O) 的相对 SE(3) 关系边编码注入 GET 的在线图注意力层，而不是替换其 RL 主干。

**待确认**：
1. 边特征注入位点先做哪种：仅 score bias（稳）还是 score+value 关系调制（强）？
2. object 表征先做哪种最小版本：单 object token + 相对变换，还是 patch-level object nodes？